# Multi-Model YOLO Validation and Reporting

Run the `run_yolo_validation_report.py` script on multiple YOLO models, collect metrics, and compare results.


In [ ]:

# 1. Set Up Environment and Install Dependencies
import os
import sys
from pathlib import Path


import torch
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image as IPImage

# Configure matplotlib for notebook
%matplotlib inline
matplotlib.rcParams['figure.max_open_warning'] = 50

print(f"Python: {sys.version}")
print(f"Torch version: {torch.__version__}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

IS_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')
if IS_COLAB:
    # Running in Google Colab
    BASE_DIR = Path('/computer_vision_yolo')
    sys.path.append("/content/Drive/MyDrive/ksu_yolo_2025/computer_vision_yolo/yolo_test")

else:
    # Running locally
    BASE_DIR = Path.cwd().parent
    sys.path.append("/computer_vision_yolo/yolo_test")

In [ ]:
#  ! cd /content/Drive/MyDrive/ksu_yolo_2025 && git clone https://github.com/m3mahdy/computer_vision_yolo

In [ ]:
# ! cd {BASE_DIR} && pip install -r requirements.txt --quiet

In [ ]:
# download limited dataset
# !mkdir {DATASET_BASE_DIR}
# !cd {BASE_DIR}/dataset && cp 8_download_extract_other_datasets.py {DATASET_BASE_DIR} && cd {DATASET_BASE_DIR} && python 8_download_extract_other_datasets.py


In [ ]:

# 1. Set Up Environment and Install Dependencies
import os
import sys
from pathlib import Path

import torch
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image as IPImage

# Configure matplotlib for notebook
%matplotlib inline
matplotlib.rcParams['figure.max_open_warning'] = 50

print(f"Python: {sys.version}")
print(f"Torch version: {torch.__version__}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

IS_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')
if IS_COLAB:
    # Running in Google Colab
    BASE_DIR = Path('/computer_vision_yolo')
else:
    # Running locally
    BASE_DIR = Path.cwd().parent


PROJECT_ROOT = BASE_DIR
SCRIPT_PATH = PROJECT_ROOT / "yolo_test" / "run_yolo_validation_report.py"

# Add project root to path for imports
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(PROJECT_ROOT / "yolo_test") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "yolo_test"))

print(f"Project root: {PROJECT_ROOT}")
print(f"Script path: {SCRIPT_PATH}")


# Dataset Selection
# Option 1: Full dataset (~100k images) - for final optimization: "bdd100k_yolo"
# Option 2: Limited dataset (representative samples) - for quick tuning: "bdd100k_yolo_limited"
DATASET_NAME = 'bdd100k_yolo_tiny'
DATASET_SPLT = 'test'  # 'train', 'val', or 'test'

BATCH_SIZE = 256
import sys


IS_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')
if IS_COLAB:
    # Running in Google Colab
    sys.path.append("/content/Drive/MyDrive/ksu_yolo_2025/computer_vision_yolo/yolo_test")

else:
    # Running locally
    sys.path.append("/computer_vision_yolo/yolo_test")

# 2. Import validation functions from script
from run_yolo_validation_report import run_validation_pipeline, visualize_predictions
print("✓ Successfully imported validation functions")

# 3. Method Loop over models, run validation, and collect metrics
results_summary = []
validation_results = {}

def test_model(models_configs):

    for cfg in models_configs:
        print("=" * 80)
        print(f"Running model: {cfg['name']} | dataset={DATASET_NAME} | split={DATASET_SPLT} | IoU={cfg['iou']}")
        print("=" * 80)

        try:
            result = run_validation_pipeline(
                model_name=cfg["name"],
                dataset_name=DATASET_NAME,
                split=DATASET_SPLT,
                iou_threshold=cfg["iou"],
                base_dir=PROJECT_ROOT,
                use_wandb=True,
                save_reports=True,
                batch_size=BATCH_SIZE,
            )

            validation_results[cfg["name"]] = result

            overall = result["metrics"]["overall"]
            yolo_overall = result["metrics"]["yolo_metrics"]

            results_summary.append({
                "model_name": cfg["name"],
                "dataset": DATASET_NAME,
                "split": DATASET_SPLT,
                "iou": cfg["iou"],
                "precision_confusion": overall["precision"],
                "recall_confusion": overall["recall"],
                "f1_confusion": overall["f1"],
                "precision_yolo": yolo_overall["precision"],
                "recall_yolo": yolo_overall["recall"],
                "map50": yolo_overall["map50"],
                "map50_95": yolo_overall["map50_95"],
                "params_m": result["model_info"]["params"] / 1e6,
                "size_mb": result["model_info"]["size(MB)"],
                "fps": result["metrics"]["fps"],
                "status": "ok",
                "run_dir": str(result["run_dir"]),
            })

        except Exception as e:
            print(f"⚠️ Model {cfg['name']} failed: {e}")
            results_summary.append({
                "model_name": cfg["name"],
                "dataset": cfg["dataset"],
                "split": cfg["split"],
                "iou": cfg["iou"],
                "status": "error",
            })

Python: 3.12.3 (v3.12.3:f6650f9ad7, Apr  9 2024, 08:18:47) [Clang 13.0.0 (clang-1300.0.29.30)]
Torch version: 2.9.1
Device: cpu
Project root: /Users/mahdy/projects/computer_vision_yolo
Script path: /Users/mahdy/projects/computer_vision_yolo/yolo_test/run_yolo_validation_report.py
✓ Successfully imported validation functions


In [ ]:
# import os, signal
# os.kill(os.getpid(), signal.SIGKILL)


# 4. Select model configurations to test

In [2]:
MODEL_CONFIGS = [
    {"name": "yolov10n",  "iou": 0.5},
]

test_model(MODEL_CONFIGS)

Running model: yolov10n | dataset=bdd100k_yolo_tiny | split=test | IoU=0.5
✓ Device: cpu
✓ W&B logging enabled


wandb: Currently logged in as: m3mahdy (m3mahdy-king-saud-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



✓ Weights & Biases initialized: yolov10n_bdd100k_yolo_tiny_test_20251127_014815
✓ Dataset loaded
  Total images: 434
  Images with labels: 434
  Label files: 434

✓ Metadata loaded: test_metadata.json
  Images with attributes: 434
✓ Model loaded from /Users/mahdy/projects/computer_vision_yolo/models/yolov10n/yolov10n.pt
YOLOv10n summary: 223 layers, 2,775,520 parameters, 0 gradients, 8.7 GFLOPs

📊 Model Information:
  Model: yolov10n
  Classes in model: 80
  Task: detect
  Parameters: 2.8M
  Model Size: 5.6 MB
  FLOPs (640x640): 8.74 GFLOPs
  Model Size: 5.6 MB

Running YOLO validation...
Ultralytics 8.3.229 🚀 Python-3.12.3 torch-2.9.1 CPU (Apple M3)
YOLOv10n summary (fused): 102 layers, 2,299,264 parameters, 0 gradients, 6.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 153.0±73.6 MB/s, size: 47.9 KB)
val: Scanning /Users/mahdy/projects/computer_vision_yolo/bdd100k_yolo_tiny/labels/test.cache... 434 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 434/434 1.5Mit/s 0.0s0

Generating comparisons: 100%|██████████| 6/6 [00:03<00:00,  1.77it/s]


✓ Generated 6 comparison images
  Saved to: /Users/mahdy/projects/computer_vision_yolo/yolo_test/runs/yolov10n_testing_20251127_014815/sample_comparisons
✓ COMPREHENSIVE REPORT GENERATED (script)
PDF Report: /Users/mahdy/projects/computer_vision_yolo/yolo_test/runs/yolov10n_testing_20251127_014815/report.pdf
JSON Metrics: /Users/mahdy/projects/computer_vision_yolo/yolo_test/runs/yolov10n_testing_20251127_014815/metrics_data.json



✓ Weights & Biases run completed successfully

🧹 Cleaning up model from memory...
✓ Model removed from memory


In [ ]:
MODEL_CONFIGS = [
    {"name": "yolov8m",  "iou": 0.5},
]

test_model(MODEL_CONFIGS)

In [ ]:
MODEL_CONFIGS = [
    {"name": "yolov8n",  "iou": 0.5},
]

test_model(MODEL_CONFIGS)

In [ ]:

MODEL_CONFIGS = [

    {"name": "yolov8s",  "iou": 0.5},

]


test_model(MODEL_CONFIGS)


In [ ]:

MODEL_CONFIGS = [

    {"name": "yolov8m",  "iou": 0.5},

]

test_model(MODEL_CONFIGS)

In [ ]:
MODEL_CONFIGS = [


    {"name": "yolov8l",  "iou": 0.5},

]


test_model(MODEL_CONFIGS)

In [ ]:
MODEL_CONFIGS = [

    {"name": "yolov8x",  "iou": 0.5},
]


test_model(MODEL_CONFIGS)

In [ ]:
# BATCH_SIZE = 128

MODEL_CONFIGS = [
  {"name": "yolov8l",  "iou": 0.5},
  {"name": "yolov8x",  "iou": 0.5},

]

test_model(MODEL_CONFIGS)


In [ ]:
# BATCH_SIZE = 32

MODEL_CONFIGS = [
  {"name": "yolov9e",  "iou": 0.5},
]

test_model(MODEL_CONFIGS)


In [ ]:
# BATCH_SIZE = 128

MODEL_CONFIGS = [
  {"name": "yolov9c",  "iou": 0.5},
]

test_model(MODEL_CONFIGS)


In [ ]:
# BATCH_SIZE = 128

MODEL_CONFIGS = [

  {"name": "yolov10x",  "iou": 0.5},


]

test_model(MODEL_CONFIGS)

In [ ]:
# BATCH_SIZE = 128

MODEL_CONFIGS = [

  {"name": "yolo11x",  "iou": 0.5},
]

test_model(MODEL_CONFIGS)

In [ ]:
# BATCH_SIZE = 128

MODEL_CONFIGS = [
  {"name": "yolo12x",  "iou": 0.5},
]

test_model(MODEL_CONFIGS)

In [ ]:
# BATCH_SIZE = 256

MODEL_CONFIGS = [
  {"name": "yolov9t",  "iou": 0.5},

]

test_model(MODEL_CONFIGS)

In [ ]:
MODEL_CONFIGS = [

  {"name": "yolov9s",  "iou": 0.5},
]

test_model(MODEL_CONFIGS)

In [ ]:
# BATCH_SIZE = 256

MODEL_CONFIGS = [
  {"name": "yolov10n",  "iou": 0.5},

]

test_model(MODEL_CONFIGS)

In [ ]:
# BATCH_SIZE = 256

MODEL_CONFIGS = [

  {"name": "yolov10s",  "iou": 0.5},
]

test_model(MODEL_CONFIGS)

In [ ]:
# BATCH_SIZE = 256

MODEL_CONFIGS = [
  {"name": "yolo11n",  "iou": 0.5},

]

test_model(MODEL_CONFIGS)

In [ ]:
# BATCH_SIZE = 256

MODEL_CONFIGS = [

  {"name": "yolo11s",  "iou": 0.5},
]

test_model(MODEL_CONFIGS)

In [ ]:
BATCH_SIZE = 256

MODEL_CONFIGS = [
  {"name": "yolo12n",  "iou": 0.5},

]

test_model(MODEL_CONFIGS)

In [ ]:
# BATCH_SIZE = 256

MODEL_CONFIGS = [

  {"name": "yolo12s",  "iou": 0.5},
]

test_model(MODEL_CONFIGS)

In [ ]:
# BATCH_SIZE = 256

MODEL_CONFIGS = [
  {"name": "yolov9m",  "iou": 0.5},

]

test_model(MODEL_CONFIGS)

In [ ]:
# BATCH_SIZE = 256

MODEL_CONFIGS = [
  {"name": "yolov10m",  "iou": 0.5},

]

test_model(MODEL_CONFIGS)

In [ ]:
# BATCH_SIZE = 128

MODEL_CONFIGS = [
  {"name": "yolo11m",  "iou": 0.5},

]

test_model(MODEL_CONFIGS)

In [ ]:
# BATCH_SIZE = 128

MODEL_CONFIGS = [
  {"name": "yolo12m",  "iou": 0.5},

]

test_model(MODEL_CONFIGS)